This notebook will be used to test two different scheduling heuristics.

In [46]:
import pandas as pd
import numpy as np

# System configuration: 4 conveyors, each with one shipping lane
N_CONVEYORS = 4

# 8 distinct shape types that the conveyor vision system can recognize
N_SHAPE_TYPES = 8
SHAPE_NAMES = ['circle', 'pentagon', 'trapezoid', 'triangle', 'star', 'moon', 'heart', 'cross']

In [47]:
# Load the CSV files generated by the data generator notebook
# Each row is one order; columns are ragged (NaN-padded) since orders have varying numbers of item types
item_types_df = pd.read_csv('ranDataGen/order_itemtypes.csv', header=None)
quantities_df = pd.read_csv('ranDataGen/order_quantities.csv', header=None)

# Build a structured list of orders
# Each order becomes a list of (item_type_index, quantity) tuples
# Processing time for an order = total number of items (sum of quantities)
orders = []
processing_times = []

for i in range(len(item_types_df)):
    # Drop NaN values from ragged rows and convert to int
    types = item_types_df.iloc[i].dropna().astype(int).tolist()
    qtys = quantities_df.iloc[i].dropna().astype(int).tolist()
    order = list(zip(types, qtys))
    orders.append(order)
    # Processing time = total quantity of items in the order
    processing_times.append(sum(qtys))

# Print summary for verification
print(f"Number of orders: {len(orders)}")
print(f"Processing times: {processing_times}")
print(f"Total items: {sum(processing_times)}")
print()
for i, order in enumerate(orders):
    print(f"Order {i}: items={order}, processing_time={processing_times[i]}")

Number of orders: 4
Processing times: [5, 5, 7, 2]
Total items: 19

Order 0: items=[(3, 3), (1, 2)], processing_time=5
Order 1: items=[(2, 3), (3, 1), (0, 1)], processing_time=5
Order 2: items=[(3, 3), (2, 3), (0, 1)], processing_time=7
Order 3: items=[(1, 1), (2, 1)], processing_time=2


In [48]:
# Parallel machine list-scheduling heuristic
# Treats each conveyor as a parallel machine and assigns orders one at a time
# to the least-loaded conveyor, balancing the workload across all conveyors
def schedule_orders(processing_times, n_conveyors, reverse=False):
    """
    Assign orders to conveyors using list scheduling.
    reverse=False: SPT (shortest processing time first)
    reverse=True: LPT (longest processing time first)
    """
    # Sort order indices by processing time (ascending for SPT, descending for LPT)
    sorted_indices = sorted(range(len(processing_times)),
                            key=lambda i: processing_times[i],
                            reverse=reverse)

    # Track the current load (total items assigned) on each conveyor
    loads = [0] * n_conveyors
    assignment = {c: [] for c in range(n_conveyors)}

    # Greedily assign each order to the conveyor with the smallest current load
    for order_idx in sorted_indices:
        # Find conveyor with minimum load (ties broken by lowest index)
        min_load = min(loads)
        conveyor = loads.index(min_load)
        assignment[conveyor].append(order_idx)
        loads[conveyor] += processing_times[order_idx]

    return assignment

In [49]:
# Build per-order rows with conveyor assignment (1-based)
# and write to CSV in the format expected by the FlexSim conveyor model
def build_output_df(assignment, orders, n_conveyors):
    """Build a DataFrame with one row per order, showing item counts and conveyor assignment."""
    rows = []
    for c in range(n_conveyors):
        for order_idx in assignment[c]:
            counts = [0] * N_SHAPE_TYPES
            for item_type, qty in orders[order_idx]:
                counts[item_type] += qty
            rows.append([c + 1] + counts)  # 1-based conveyor numbering

    columns = ['conv_num'] + SHAPE_NAMES
    return pd.DataFrame(rows, columns=columns)

def save_conveyor_csv(df, filepath):
    """Save CSV matching the example format: no-space header, space-after-comma data."""
    with open(filepath, 'w') as f:
        # Header row: comma-separated with no spaces (matches FlexSim import format)
        f.write(','.join(df.columns) + '\n')
        # Data rows: comma-space separated (matches example input format)
        for _, row in df.iterrows():
            f.write(', '.join(str(int(v)) for v in row) + '\n')
    print(f"Saved to {filepath}")

# Shortest Processing Time 

In [50]:
# Run SPT heuristic: sort orders from shortest to longest processing time,
# then assign each order to the least-loaded conveyor
spt_assignment = schedule_orders(processing_times, N_CONVEYORS, reverse=False)

# Display which orders were assigned to each conveyor and their total load
print("SPT (Shortest Processing Time) Assignment:")
for c in range(N_CONVEYORS):
    order_list = spt_assignment[c]
    load = sum(processing_times[i] for i in order_list)
    print(f"  Conveyor {c}: orders={order_list}, load={load}")

# Makespan = the maximum load across all conveyors (determines total completion time)
spt_makespan = max(sum(processing_times[i] for i in spt_assignment[c]) for c in range(N_CONVEYORS))
print(f"\nSPT Makespan: {spt_makespan}")

SPT (Shortest Processing Time) Assignment:
  Conveyor 0: orders=[3], load=2
  Conveyor 1: orders=[0], load=5
  Conveyor 2: orders=[1], load=5
  Conveyor 3: orders=[2], load=7

SPT Makespan: 7


In [51]:
# Build the conveyor input table for SPT and save as CSV
# Each row represents one order with its shape counts and assigned conveyor (1-based)
spt_df = build_output_df(spt_assignment, orders, N_CONVEYORS)
print("SPT Conveyor Input:")
print(spt_df.to_string(index=False))
print()
save_conveyor_csv(spt_df, 'frConveryor/SPT_input.csv')

SPT Conveyor Input:
 conv_num  circle  pentagon  trapezoid  triangle  star  moon  heart  cross
        1       0         1          1         0     0     0      0      0
        2       0         2          0         3     0     0      0      0
        3       1         0          3         1     0     0      0      0
        4       1         0          3         3     0     0      0      0

Saved to frConveryor/SPT_input.csv


# Longest Processing Time

In [52]:
# Run LPT heuristic: sort orders from longest to shortest processing time,
# then assign each order to the least-loaded conveyor
# LPT generally produces better (lower) makespans than SPT for parallel machine scheduling
lpt_assignment = schedule_orders(processing_times, N_CONVEYORS, reverse=True)

# Display which orders were assigned to each conveyor and their total load
print("LPT (Longest Processing Time) Assignment:")
for c in range(N_CONVEYORS):
    order_list = lpt_assignment[c]
    load = sum(processing_times[i] for i in order_list)
    print(f"  Conveyor {c}: orders={order_list}, load={load}")

# Makespan = the maximum load across all conveyors (determines total completion time)
lpt_makespan = max(sum(processing_times[i] for i in lpt_assignment[c]) for c in range(N_CONVEYORS))
print(f"\nLPT Makespan: {lpt_makespan}")

LPT (Longest Processing Time) Assignment:
  Conveyor 0: orders=[2], load=7
  Conveyor 1: orders=[0], load=5
  Conveyor 2: orders=[1], load=5
  Conveyor 3: orders=[3], load=2

LPT Makespan: 7


In [53]:
# Build the conveyor input table for LPT and save as CSV
# Each row represents one order with its shape counts and assigned conveyor (1-based)
lpt_df = build_output_df(lpt_assignment, orders, N_CONVEYORS)
print("LPT Conveyor Input:")
print(lpt_df.to_string(index=False))
print()
save_conveyor_csv(lpt_df, 'frConveryor/LPT_input.csv')

LPT Conveyor Input:
 conv_num  circle  pentagon  trapezoid  triangle  star  moon  heart  cross
        1       1         0          3         3     0     0      0      0
        2       0         2          0         3     0     0      0      0
        3       1         0          3         1     0     0      0      0
        4       0         1          1         0     0     0      0      0

Saved to frConveryor/LPT_input.csv


In [54]:
# Compare the two heuristics side-by-side
# Makespan is the key metric: it represents the time the last conveyor finishes
# A lower makespan means better load balancing across conveyors
print("=" * 50)
print("COMPARISON SUMMARY")
print("=" * 50)
print(f"{'Metric':<25} {'SPT':>10} {'LPT':>10}")
print("-" * 50)
print(f"{'Makespan':<25} {spt_makespan:>10} {lpt_makespan:>10}")
spt_total = sum(processing_times)
lpt_total = sum(processing_times)
print(f"{'Total items':<25} {spt_total:>10} {lpt_total:>10}")
print("-" * 50)
if lpt_makespan <= spt_makespan:
    print("LPT produces a better or equal makespan.")
else:
    print("SPT produces a better makespan.")

COMPARISON SUMMARY
Metric                           SPT        LPT
--------------------------------------------------
Makespan                           7          7
Total items                       19         19
--------------------------------------------------
LPT produces a better or equal makespan.


# Experiment Setup Guide

In [55]:
# ─────────────────────────────────────────────────────────────────────────────
# Part A: Load tote data and build tote contents
# ─────────────────────────────────────────────────────────────────────────────

totes_df = pd.read_csv('ranDataGen/orders_totes.csv', header=None)

tote_contents = {}       # {tote_id: [{'order', 'item_type', 'quantity'}, ...]}
orders_with_totes = set()

for order_idx in range(len(item_types_df)):
    if order_idx >= len(totes_df):
        continue
    types = item_types_df.iloc[order_idx].dropna().astype(int).tolist()
    qtys  = quantities_df.iloc[order_idx].dropna().astype(int).tolist()
    tids  = totes_df.iloc[order_idx].dropna().astype(int).tolist()
    for item_type, quantity, tote_id in zip(types, qtys, tids):
        tote_contents.setdefault(tote_id, []).append({
            'order': order_idx, 'item_type': item_type, 'quantity': quantity
        })
        orders_with_totes.add(order_idx)

multi_order_totes = {
    tid for tid, entries in tote_contents.items()
    if len({e['order'] for e in entries}) > 1
}

orders_without_totes = sorted(set(range(len(orders))) - orders_with_totes)

print(f"Loaded {len(tote_contents)} totes from orders_totes.csv")
print(f"Multi-order totes: {sorted(multi_order_totes)}")
if orders_without_totes:
    print(f"WARNING: Orders {orders_without_totes} have no tote assignments "
          f"-- will auto-complete in sequence")


# ─────────────────────────────────────────────────────────────────────────────
# Part B: Greedy tote sequencing (adapted from greedy_tote_simulation.py)
# ─────────────────────────────────────────────────────────────────────────────

def compute_greedy_tote_sequence(assignment, tote_contents, orders,
                                 processing_times, n_conveyors):
    """Pick totes by greedy marginal gain: most items to active orders first.
    Ties broken by unblocking value, then smallest tote ID."""
    orders_in_totes = {e['order'] for ents in tote_contents.values() for e in ents}

    remaining = {}
    for oid in range(len(orders)):
        if oid in orders_in_totes:
            remaining[oid] = {}
            for it, qty in orders[oid]:
                remaining[oid][it] = remaining[oid].get(it, 0) + qty
        else:
            remaining[oid] = {}

    queues    = {c: list(assignment[c]) for c in range(n_conveyors)}
    queue_pos = [0] * n_conveyors

    def is_complete(oid):
        return all(v <= 0 for v in remaining.get(oid, {}).values())

    # Advance past initially-complete orders (e.g. no tote data)
    auto_completed = []
    for c in range(n_conveyors):
        while queue_pos[c] < len(queues[c]) and is_complete(queues[c][queue_pos[c]]):
            auto_completed.append(queues[c][queue_pos[c]])
            queue_pos[c] += 1

    def active_orders():
        return {queues[c][queue_pos[c]]
                for c in range(n_conveyors) if queue_pos[c] < len(queues[c])}

    def score_tote(tid, active):
        primary = 0
        temp = {oid: dict(d) for oid, d in remaining.items()}
        for e in tote_contents[tid]:
            oid, it, qty = e['order'], e['item_type'], e['quantity']
            if oid in active:
                useful = min(qty, max(temp.get(oid, {}).get(it, 0), 0))
                primary += useful
                if it in temp.get(oid, {}):
                    temp[oid][it] -= qty
        secondary = 0
        for c in range(n_conveyors):
            if queue_pos[c] < len(queues[c]):
                oid = queues[c][queue_pos[c]]
                if oid in active and all(v <= 0 for v in temp.get(oid, {}).values()):
                    secondary += 2 if (queue_pos[c] + 1) < len(queues[c]) else 1
        return (primary, secondary)

    remaining_totes = sorted(tote_contents.keys())
    sequence, step_details = [], []

    while remaining_totes:
        active = active_orders()
        if not active:
            for tid in remaining_totes:
                sequence.append(tid)
                step_details.append({'tote_id': tid, 'active_orders': [],
                                     'deliveries': {}, 'completed_orders': [],
                                     'constraint_violation': False})
            break

        best, best_s = None, (-1, -1)
        for tid in remaining_totes:
            s = score_tote(tid, active)
            if s > best_s or (s == best_s and (best is None or tid < best)):
                best_s, best = s, tid

        info = {'tote_id': best, 'active_orders': sorted(active),
                'deliveries': {}, 'completed_orders': [],
                'constraint_violation': False}

        tote_ords = {e['order'] for e in tote_contents[best]}
        if len(tote_ords) > 1 and not tote_ords.issubset(active):
            info['constraint_violation'] = True

        for e in tote_contents[best]:
            oid, it, qty = e['order'], e['item_type'], e['quantity']
            if oid in active and oid in remaining and it in remaining[oid]:
                delivered = min(qty, max(remaining[oid][it], 0))
                if delivered > 0:
                    info['deliveries'].setdefault(oid, {})
                    info['deliveries'][oid][it] = \
                        info['deliveries'][oid].get(it, 0) + delivered
                remaining[oid][it] -= qty

        for c in range(n_conveyors):
            while queue_pos[c] < len(queues[c]) and is_complete(queues[c][queue_pos[c]]):
                info['completed_orders'].append(queues[c][queue_pos[c]])
                queue_pos[c] += 1

        sequence.append(best)
        step_details.append(info)
        remaining_totes.remove(best)

    return sequence, step_details, auto_completed


# ─────────────────────────────────────────────────────────────────────────────
# Part C: Display function
# ─────────────────────────────────────────────────────────────────────────────

def print_experiment_guide(label, assignment, tote_contents, orders,
                           processing_times, n_conveyors):
    """Print a formatted experiment setup guide for one heuristic."""
    order_to_conv = {}
    for c in range(n_conveyors):
        for oid in assignment[c]:
            order_to_conv[oid] = c

    sequence, step_details, auto_completed = compute_greedy_tote_sequence(
        assignment, tote_contents, orders, processing_times, n_conveyors)

    W = 64
    print(f"\n{'=' * W}")
    print(f"  EXPERIMENT SETUP GUIDE  --  {label}")
    print(f"{'=' * W}")

    # ── Conveyor Queue Assignments ──
    print(f"\n  CONVEYOR QUEUE ASSIGNMENTS")
    print(f"{'-' * W}")
    for c in range(n_conveyors):
        if not assignment[c]:
            print(f"  Conveyor {c+1}:  (empty)")
        else:
            parts = [f"Order {o} ({processing_times[o]} items)" for o in assignment[c]]
            print(f"  Conveyor {c+1}:  {', '.join(parts)}")

    # ── Tote Packing List ──
    print(f"\n  TOTE PACKING LIST")
    print(f"{'-' * W}")
    for tid in sorted(tote_contents):
        tag = "  [MULTI-ORDER]" if tid in multi_order_totes else ""
        print(f"  Tote {tid}:{tag}")
        by_order = {}
        for e in tote_contents[tid]:
            by_order.setdefault(e['order'], []).append(e)
        for oid in sorted(by_order):
            conv = f"Conveyor {order_to_conv[oid]+1}" if oid in order_to_conv else "(?)"
            items = "  +  ".join(
                f"{e['quantity']}x {SHAPE_NAMES[e['item_type']]}" for e in by_order[oid])
            print(f"    Order {oid} -> {conv}:  {items}")

    missing = sorted(o for o in order_to_conv if o not in orders_with_totes)
    if missing:
        print(f"\n  NOTE: Orders {missing} have no tote assignments")

    # ── Recommended Tote Loading Sequence ──
    print(f"\n  RECOMMENDED TOTE LOADING SEQUENCE")
    print(f"{'-' * W}")

    all_completed = set(auto_completed)
    for step, info in enumerate(step_details, 1):
        print(f"  Step {step}:  Load TOTE {info['tote_id']}")
        print(f"    Active orders: {info['active_orders']}")

        if info['constraint_violation']:
            print("    *** CONSTRAINT VIOLATION *** "
                  "Multi-order tote but not all orders active!")

        if info['deliveries']:
            print("    Delivers to:")
            for oid in sorted(info['deliveries']):
                conv = (f"Conveyor {order_to_conv[oid]+1}"
                        if oid in order_to_conv else "Conveyor (?)")
                items = info['deliveries'][oid]
                parts = "  +  ".join(
                    f"{qty}x {SHAPE_NAMES[it]}" for it, qty in sorted(items.items()))
                print(f"      {conv} (Order {oid}):  {parts}")
        else:
            print("    (no items delivered to active orders)")

        if info['completed_orders']:
            cstr = ', '.join(f'Order {o}' for o in sorted(info['completed_orders']))
            print(f"    Orders COMPLETED: [{cstr}]")
            all_completed.update(info['completed_orders'])
        print()

    all_orders = set(range(len(orders)))
    if all_completed >= all_orders:
        print(f"  All {len(orders)} orders completed.")
    else:
        incomplete = sorted(all_orders - all_completed)
        print(f"  WARNING: Orders {incomplete} did NOT complete!")
    print(f"{'=' * W}")


# ─── Run for both heuristics ────────────────────────────────────────────────

print_experiment_guide("SPT (Shortest Processing Time)",
                       spt_assignment, tote_contents, orders,
                       processing_times, N_CONVEYORS)

print_experiment_guide("LPT (Longest Processing Time)",
                       lpt_assignment, tote_contents, orders,
                       processing_times, N_CONVEYORS)

Loaded 4 totes from orders_totes.csv
Multi-order totes: [1, 2, 3]

  EXPERIMENT SETUP GUIDE  --  SPT (Shortest Processing Time)

  CONVEYOR QUEUE ASSIGNMENTS
----------------------------------------------------------------
  Conveyor 1:  Order 3 (2 items)
  Conveyor 2:  Order 0 (5 items)
  Conveyor 3:  Order 1 (5 items)
  Conveyor 4:  Order 2 (7 items)

  TOTE PACKING LIST
----------------------------------------------------------------
  Tote 0:
    Order 3 -> Conveyor 1:  1x pentagon  +  1x trapezoid
  Tote 1:  [MULTI-ORDER]
    Order 0 -> Conveyor 2:  3x triangle  +  2x pentagon
    Order 2 -> Conveyor 4:  1x circle
  Tote 2:  [MULTI-ORDER]
    Order 1 -> Conveyor 3:  3x trapezoid  +  1x circle
    Order 2 -> Conveyor 4:  3x trapezoid
  Tote 3:  [MULTI-ORDER]
    Order 1 -> Conveyor 3:  1x triangle
    Order 2 -> Conveyor 4:  3x triangle

  RECOMMENDED TOTE LOADING SEQUENCE
----------------------------------------------------------------
  Step 1:  Load TOTE 2
    Active orders: [0,